# The Watchline agent and chat

*Executed 2026-11-22 against the live Watchline graph (PLUTO 26v1, HPD registrations through 2026-05-29), Geosupport 25b and
`claude-haiku-5-5`. The model's wording changes from run to run; the data under it does not. Re-run with `make notebook-check`
(needs Neo4j, the sidecar and `ANTHROPIC_API_KEY`; each run makes about a dozen cheap model calls).*

`make chat` starts a conversation with an agent that has **one tool**, `building_profile`. This notebook drives that same
agent, with the same functions the chat uses (`build_agent`, `ask`), and shows for every question:

1. what you typed,
2. the tool call the model chose (`-> building_profile(...)`),
3. the answer, and how it lines up with the data the tool returned.

Nothing here is a finding about a landlord. Violations, problems and litigation are what was **cited, reported or brought**.
The evals that grade answers like these automatically are `make eval-agent`; this notebook is for reading, not grading.

## Setup

`connect()` is the chat's own startup: it checks the Neo4j login and the sidecar and fails with one clear line if either is down.

In [1]:
import json
import uuid

from IPython.display import Markdown, display
from langgraph.checkpoint.memory import InMemorySaver

from watchline.agent.agent import system_prompt
from watchline.agent.agent import build_agent
from watchline.agent.chat import ask, connect
from watchline.agent.config import AGENT_MODEL

client, graph, pg, close = connect()
agent = build_agent(client, graph, pg, checkpointer=InMemorySaver())
RUN = uuid.uuid4().hex[:8]
print("model:", AGENT_MODEL)

model: claude-haiku-5-5


### What the model is given

The system prompt is written in `watchline/agent/prompts/system.md`. The ontology rules are filled in from
`watchline/agent/prompts/ontology_rules.md`, the same file `CLAUDE.md` imports, so Claude Code and the agent read the same words.

In [2]:
prompt = system_prompt()
print(f"{len(prompt.splitlines())} lines. The part about reading the tool result:\n")
start = prompt.index("## Reading the tool result")
print(prompt[start:prompt.index("## Shape of a profile answer")])

73 lines. The part about reading the tool result:

## Reading the tool result

Check `status` first, then `resolution.outcome`.

- `needs_borough`: ask the user for the borough or ZIP. Do not guess.
- `invalid_input`: say you need a house number and street, and ask for one.
- `error`: say the lookup failed and this is an outage, not an answer about the address. Suggest trying again.
- `ok` with `ambiguous`: list the candidate streets and ask which was meant. No lot was chosen.
- `ok` with `not_found`: say no lot was found, give the reason from the caveats, and do not guess a nearby address.
- `ok` with `exact` or `variant`: profile the lot, unless `lot` is null. A null `lot` means the address resolved but the
  lot is not in the graph; there is no profile, so say so using the caveats. For a variant, say what was typed and what the
  city matched. If the resolution is inferred, say so.




### One helper

`chat(text, thread)` is the chat loop for one message. A *thread* is one conversation: messages on the same thread see each
other, which is how a follow-up like "what about 218?" works. It returns the turn, so the raw tool data can be inspected.

In [3]:
def chat(text, thread):
    config = {"configurable": {"thread_id": f"{RUN}-{thread}"}}
    print(f"you> {text}")
    turn = ask(agent, text, config, on_tool_call=lambda name, args: print(f"  -> {name}({json.dumps(args)})"))
    display(Markdown(turn.answer))
    return turn


def tool_view(turn):
    """The parts of the first tool result that matter, to compare with the answer."""
    out = turn.tool_results[0]
    res = out["resolution"]
    print(f"status: {out['status']}   outcome: {res['outcome']}   bbl: {res['bbl']}   in_graph: {res['in_graph']}")
    print("counts:", {c["name"]: c["value"] for c in out.get("counts", [])})
    print("BINs:", [(b["bin"], "addressed" if b["addressed"] else "") for b in out.get("buildings", [])])
    for c in out["caveats"]:
        print("caveat:", c)

## 1. A lot with three buildings, then a follow-up on the same thread

`224 54 Street, Brooklyn` is one tax lot with three HPD-registered buildings. The model should call the tool once, name the
addressed building, list the others, and keep residential units as a **lot-wide** figure.

In [4]:
multi = chat("What can you tell me about 224 54 Street in Brooklyn?", "multi")

you> What can you tell me about 224 54 Street in Brooklyn?


  -> building_profile({"address": "224 54 Street", "borough": "Brooklyn"})


**224 54 Street, Brooklyn** matched exactly to tax lot (BBL) **3008210012**. The BIN for the addressed building is 3324451.

**Lot characteristics (PLUTO release 26v1, lot-wide)**
- Year built: 1911
- Building class: C1
- Residential units: 18 (belongs to the lot, not any one building)
- Assessed total value: $814,500

**Counts**
- Tax lots: 1
- Buildings as PLUTO counts them: 3
- Buildings HPD lists in its registrations: 3
- Residential units, lot-wide: 18

**Buildings HPD lists for this lot**
1. **224 54th Street** (BIN 3324451), the addressed building. Complaint problems: Door/Window 3, Flooring/Stairs 1, Heat/Hot Water 3, Paint/Plaster 1, Unsanitary Condition 1, Water Leak 1. No violations or litigation recorded.
2. **218 54th Street** (BIN 3324448). No violations, complaint problems or litigation recorded.
3. **220 54th Street** (BIN 3324450). Violations: one each in Class A, B and C, all with status Close. Complaint problems: Door/Window 2, Heat/Hot Water 9 plus 2 rows flagged as duplicates, Plumbing 1. No litigation recorded.

**Across all three listed buildings (lot total, by type)**
- Violations: Class A 1 (Close), Class B 1 (Close), Class C 1 (Close)
- Complaint problems: Heat/Hot Water 12 (plus 2 duplicate-flagged rows, counted separately), Door/Window 5, Flooring/Stairs 1, Paint/Plaster 1, Plumbing 1, Unsanitary Condition 1, Water Leak 1
- Litigation: none recorded

**Harassment findings:** none recorded after inquest or trial for any of the three buildings. This means the records show no such finding. It does not mean the records show none were brought.

**Caveats from the records**
- Violations, complaint problems and litigation are what was cited, reported or brought. They are not findings of fault. The counts are by class or type and are not a severity score.
- Residential units and assessed value are lot-wide and do not belong to any single building.
- Problem rows flagged as duplicates are counted separately from the others.

The data the model was given, to check the answer against:

In [5]:
tool_view(multi)
data = multi.tool_results[0]
print()
print("BBL in the answer:", data["resolution"]["bbl"] in multi.answer)
for b in data["buildings"]:
    print(f"BIN {b['bin']} in the answer:", b["bin"] in multi.answer)

status: ok   outcome: exact   bbl: 3008210012   in_graph: True
counts: {'parcels': 1, 'buildings (PLUTO)': 3, 'buildings (HPD)': 3, 'residential units': 18}
BINs: [('3324451', 'addressed'), ('3324448', ''), ('3324450', '')]
caveat: Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
caveat: Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
caveat: Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.

BBL in the answer: True
BIN 3324451 in the answer: True
BIN 3324448 in the answer: True
BIN 3324450 in the answer: True


The follow-up is answered on the **same thread**. Because the first answer already holds all three buildings, the model may answer
from the conversation or call the tool again; either is fine, as long as 218 54 Street is placed on the same tax lot.

In [6]:
follow = chat("And what about the building at 218?", "multi")
print("tool calls this turn:", [c["args"] for c in follow.tool_calls] or "none (answered from the conversation)")

you> And what about the building at 218?


  -> building_profile({"address": "218 54 Street", "borough": "Brooklyn"})


**218 54th Street** (BIN 3324448) is one of the three buildings HPD lists on the same tax lot as 224 54 Street (BBL 3008210012). The lot's characteristics are the same as before: built 1911, class C1, 18 residential units and $814,500 assessed, all lot-wide rather than specific to this building.

**Records for 218 54th Street**
- Violations: none recorded
- Complaint problems: none recorded
- Litigation: none recorded
- Harassment findings after inquest or trial: none recorded. This means the records show no such finding, not that none were brought.

For comparison, the other two listed buildings on the lot have recorded items. 220 54th Street has violations and complaint problems, and 224 54th Street has complaint problems. The counts for each building are in my previous answer.

**Caveats from the records**
- Violations, complaint problems and litigation are what was cited, reported or brought. They are not findings of fault.
- Residential units and assessed value belong to the lot, not to 218 alone.
- The records list the building's address as 218 54th Street. The lot's address reads 224 54 Street.

tool calls this turn: [{'address': '218 54 Street', 'borough': 'Brooklyn'}]


## 2. Two building counts that disagree

PLUTO says 2 buildings at `17 Battery Place`; HPD lists 1. Neither is "the number of buildings". The answer should give both, each
labelled with its source.

In [7]:
battery = chat("How many buildings are at 17 Battery Place, Manhattan?", "battery")
tool_view(battery)

you> How many buildings are at 17 Battery Place, Manhattan?


  -> building_profile({"address": "17 Battery Place", "borough": "Manhattan"})


**17 Battery Place, Manhattan** matched exactly to tax lot (BBL) **1000157501**.

**Building count: two different numbers, from two sources**
- **PLUTO (release 26v1) counts 2 buildings** on the lot.
- **HPD registrations list 1 building** (BIN 1082634). That listing's address is **1 West Street**, and it is the building the address resolved to.

Neither number alone answers "how many buildings." The two sources differ on many lots.

**Lot characteristics (lot-wide, PLUTO release 26v1)**
- Year built: 1902
- Building class: RM
- Residential units: 500 (lot-wide, not per building)
- Assessed total value: $118,120,050 (lot-wide)

**Cited, reported or brought, for the HPD-listed building (BIN 1082634)**
- Violations by class and status:
  - Class A: 3 closed, 1 open
  - Class B: 8 closed
  - Class C: 7 closed, 1 open
- Complaint problems by category: Unsanitary condition 17; Flooring/stairs 2; Heat/hot water 2; Water leak 2; Door/window 1; Paint/plaster 1; Plumbing 1
- Litigation by case type and status: Tenant Action, closed, 2; Tenant Action/Harrassment, closed, 1
- Harassment findings after inquest or trial: 0 recorded

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought. They are not findings of fault. Counts are by class or type and are not a severity score.
- Residential units and assessed value belong to the lot, not to any single building.
- PLUTO's count of 2 and HPD's count of 1 come from different sources. Neither is the definitive building count.
- Only "After Inquest" or "After Trial" is a harassment finding. The recorded count of 0 means none of that kind is recorded. It does not show that no harassment occurred. Cases with no recorded finding are not findings either way.
- The totals above cover only the one building HPD lists. The second PLUTO building has no HPD registration in these records, so its violations, problems and litigation are not shown here.

status: ok   outcome: exact   bbl: 1000157501   in_graph: True
counts: {'parcels': 1, 'buildings (PLUTO)': 2, 'buildings (HPD)': 1, 'residential units': 500}
BINs: [('1082634', 'addressed')]
caveat: Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
caveat: Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
caveat: Two building counts: PLUTO says 2, HPD registrations list 1. They are different sources and differ on many lots; neither alone is 'the number of buildings'.
caveat: Only a finding of 'After Inquest' or 'After Trial' is a harassment finding. 'No Harassment' is a recorded finding of none, and many cases have no finding recorded; those are not findings either way.


## 3. A lot HPD does not register

`70 Broad Street` is a tax lot with no HPD-registered building. The records hold no violations, complaints or litigation for it,
**because they exist only for registered buildings**. That is not the same as "none". The answer should say "no building recorded in
the HPD registrations", not "no buildings" and not "no violations".

In [8]:
broad = chat("Tell me about 70 Broad Street, Manhattan.", "broad")
tool_view(broad)

you> Tell me about 70 Broad Street, Manhattan.


  -> building_profile({"address": "70 Broad Street", "borough": "Manhattan"})


**70 Broad Street, Manhattan** matched exactly to tax lot (BBL) **1000110015**. The records show no HPD-registered building for this lot, so there are no violation, complaint or litigation counts to report.

**Lot characteristics (lot-wide)**
- Year built: 1908 (PLUTO, release 26v1)
- Building class: O8 (PLUTO, release 26v1)
- Residential units: 3 (PLUTO, release 26v1, lot-wide)
- Assessed total value: $4,751,550 (PLUTO, release 26v1, lot-wide)

**Counts**
- Tax lots: 1 (Geosupport, release 25b)
- Buildings as PLUTO counts them: 1 (PLUTO, release 26v1)
- Buildings HPD lists: 0 (HPD registrations, data through 2026-05-29)
- Residential units: 3, lot-wide (PLUTO)

The lookup also returned BIN 1000036 for this address. HPD does not list that building in its registrations, so I cannot tie any HPD record to it.

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought. They are not findings of fault.
- Residential units and assessed value belong to the lot, not to any single building.
- No building is recorded in the HPD registrations. That does not mean the lot has no buildings. Violations, problems and litigation exist only for HPD-registered buildings, so their absence here does not mean there were none.

status: ok   outcome: exact   bbl: 1000110015   in_graph: True
counts: {'parcels': 1, 'buildings (PLUTO)': 1, 'buildings (HPD)': 0, 'residential units': 3}
BINs: []
caveat: Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
caveat: Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
caveat: No building recorded in the HPD registrations. This is not a statement that the lot has no buildings; violations, problems and litigation exist only for HPD-registered buildings.


## 4. Not enough to resolve: a missing borough, then the answer

`224 54 Street` exists in more than one borough. The tool returns `needs_borough`, and the model should ask, not guess. The reply
"Brooklyn" arrives on the same thread, and the model completes the lookup.

In [9]:
ask1 = chat("224 54 Street", "borough")
print("tool status:", ask1.tool_results[0]["status"] if ask1.tool_results else "tool not called")

you> 224 54 Street


  -> building_profile({"address": "224 54 Street"})


I can look that up, but I need the borough or ZIP code for 224 54 Street first. There are several 54th Streets in New York City, so I don't want to guess. Which borough is it in (Manhattan, Brooklyn, Queens, the Bronx or Staten Island), or what is the ZIP code?

tool status: needs_borough


In [10]:
ask2 = chat("Brooklyn", "borough")

you> Brooklyn


  -> building_profile({"address": "224 54 Street", "borough": "Brooklyn"})


**Match:** 224 54 Street, Brooklyn, resolved as an exact match to tax lot (BBL) 3008210012.

**Buildings:** HPD registrations list 3 buildings (BINs) for this lot:
- **3324451, 224 54th Street (the addressed building)**
- 3324448, 218 54th Street
- 3324450, 220 54th Street

**Lot characteristics (lot-wide, from PLUTO release 26v1):**
- Year built: 1911
- Building class: C1
- Residential units: 18
- Assessed total value: $814,500

**Counts:**
- Tax lots: 1
- Buildings as PLUTO counts them: 3
- Buildings HPD lists: 3
- Residential units, lot-wide: 18

**Cited, reported or brought, by building:**

- **224 54th Street (BIN 3324451):** No violations. Problems: Heat/Hot Water 3, Door/Window 3, Flooring/Stairs 1, Paint/Plaster 1, Unsanitary Condition 1, Water Leak 1. No litigation.
- **218 54th Street (BIN 3324448):** No violations, problems or litigation recorded.
- **220 54th Street (BIN 3324450):** Violations: Class A 1, Class B 1, Class C 1, all with status Close. Problems: Heat/Hot Water 9 plus 2 rows flagged as duplicates, Door/Window 2, Plumbing 1. No litigation.

**Across all 3 listed buildings (not per building):** Violations are Class A 1, B 1 and C 1, all Close. Problems total 25 rows, including 2 duplicate-flagged rows. Litigation: none.

**Harassment findings (after inquest or trial):** 0 for each building. This means no such finding is recorded in these records. It does not mean the records were checked and found clean.

**Caveats:**
- Violations, complaint problems and litigation are what was cited, reported or brought. They are not findings of fault, and the counts are by class or type, not a severity score.
- Residential units (18) and assessed value ($814,500) belong to the whole lot, not to any one building.
- Problem rows flagged as duplicates are counted separately from the rest.
- HPD registration data runs through 2026-05-29.

## 5. An ambiguous street and an address that is not there

`20 Smith` matches several streets, so the model should list them and ask which was meant. It must not profile a tax lot.
`1 Fake Streeet` matches nothing: the answer should say no lot was found and must not offer a nearby one.

In [11]:
smith = chat("Tell me about 20 Smith, Brooklyn", "smith")
print("tool outcome:", smith.tool_results[0]["resolution"]["outcome"], "  candidates:", smith.tool_results[0]["resolution"]["candidates"][:4])

you> Tell me about 20 Smith, Brooklyn


  -> building_profile({"address": "20 Smith", "borough": "Brooklyn"})


The address "20 Smith" in Brooklyn is ambiguous, so I haven't matched it to a tax lot. The records show three similar street names:

1. **Smith Jr Memorial Way**
2. **Smith Street**
3. **Smiths Lane**

Which one did you mean? If you know the ZIP code, that would also help narrow it down.

tool outcome: ambiguous   candidates: ['SMITH JR MEM WAY', 'SMITH STREET', 'SMITHS LANE']


In [12]:
fake = chat("Tell me about 1 Fake Streeet, Brooklyn", "fake")
print("tool outcome:", fake.tool_results[0]["resolution"]["outcome"])

you> Tell me about 1 Fake Streeet, Brooklyn


  -> building_profile({"address": "1 Fake Streeet", "borough": "Brooklyn"})


I couldn't find a tax lot for **1 Fake Streeet, Brooklyn**. The address resolved as "not found" (Geosupport return code 11), and the city's lookup only tried the spelling you entered. The records have no lot, building, violation, complaint or litigation data for it.

I haven't guessed at a nearby address. The spelling "Streeet" may be a typo. If you meant a real Brooklyn street, send me the house number and street name as you intended them, and I'll look that up.

tool outcome: not_found


## 6. A harassment finding, and what is not one

Only a litigation case that ended **After Inquest** or **After Trial** is a harassment finding. A case *of type* "Tenant
Action/Harrassment" is a case that was brought. `21-08 Nameoke Avenue` has exactly one finding; the answer should report that count in
those words and describe the other cases as brought, not found.

In [13]:
nameoke = chat("21-08 Nameoke Avenue in Queens: has there been a harassment finding there?", "nameoke")
for b in nameoke.tool_results[0]["buildings"]:
    print(f"BIN {b['bin']}: harassment findings (After Inquest or After Trial) = {b['harassment_findings_after_inquest_or_trial']}")
    print("   litigation by type and status:", b["litigation"])

you> 21-08 Nameoke Avenue in Queens: has there been a harassment finding there?


  -> building_profile({"address": "21-08 Nameoke Avenue", "borough": "Queens"})


**Yes, the records show one harassment finding for this building.** The HPD litigation data records 1 case with an "After Inquest" or "After Trial" outcome. Those are the only outcomes the records count as harassment findings. The tool does not show the case number, date or case status for that finding, so check the case record for those details.

**The match**
- Address: 21-08 Nameoke Avenue, Queens
- Outcome: exact match
- Tax lot (BBL): 4155250040
- Building (BIN): 4297814

**Buildings**
HPD lists one building for this lot, 4297814 at 21-08 Nameoke Avenue, which is the addressed building.

**Lot characteristics (lot-wide, PLUTO release 26v1)**
- Year built: 1920
- Building class: C3
- Residential units: 4 (lot-wide)
- Assessed total value: $190,800 (lot-wide)

**Counts**
- Tax lots: 1
- Buildings as PLUTO counts them: 1
- Buildings HPD lists: 1
- Residential units: 4 (lot-wide)

**What was cited or brought (the one building)**
- Violations: Class A, 5 (closed); Class B, 1 (closed) and 1 (open); Class C, 1 (closed); Class I, 1 (closed)
- Complaint problems: 31 unsanitary condition, 8 paint/plaster, 7 electric, 4 door/window, 4 heat/hot water, 3 appliance, 3 general, 2 flooring/stairs, 2 plumbing, 1 water leak
- Litigation: 4 "Tenant Action/Harrassment" cases (all closed), 1 "Access Warrant - Non-Lead" case (closed)
- Harassment findings after inquest or trial: 1

**Caveats**
- Violations, complaint problems and litigation are what was cited, reported or brought. They are not findings of fault.
- Counts are by class or type. They are not a severity score.
- The 4 harassment cases are what was brought. Only the 1 "After Inquest" or "After Trial" outcome counts as a finding. The records do not show the outcome of the other cases. Many cases have no recorded finding, and that is not a finding either way.
- Residential units and assessed value belong to the lot, not to the building.

BIN 4297814: harassment findings (After Inquest or After Trial) = 1
   litigation by type and status: [{'caseType': 'Access Warrant - Non-Lead', 'caseStatus': 'CLOSED', 'n': 1}, {'caseType': 'Tenant Action/Harrassment', 'caseStatus': 'CLOSED', 'n': 4}]


## 7. Questions the agent should refuse

**Judging fault.** The records report what was cited, reported or brought. They do not say a landlord is bad, and the agent must not
say so, or rank buildings, or add counts into a score.

In [14]:
slum = chat("Is the landlord at 224 54 Street, Brooklyn a slumlord?", "slum")

you> Is the landlord at 224 54 Street, Brooklyn a slumlord?


  -> building_profile({"address": "224 54 Street", "borough": "Brooklyn"})


I can't say whether the landlord is a slumlord. That is a judgment, and the records don't make it. They show what was cited, reported or brought against the buildings on this lot. The lookup also didn't return the owner or registration contact, so this answer covers the buildings, not a named landlord.

**The match**
- 224 54 Street, Brooklyn resolved exactly to tax lot (BBL) 3008210012.

**Buildings (BINs)**
HPD lists three buildings for this lot. The addressed building is marked.
- 3324451, 224 54th Street (addressed building)
- 3324448, 218 54th Street
- 3324450, 220 54th Street

PLUTO counts 3 buildings on the lot, and HPD lists 3 BINs. The lot is not flagged as having more buildings than these.

**Lot characteristics** (lot-wide, not per building)
- Year built: 1911 (PLUTO, release 26v1)
- Building class: C1 (PLUTO)
- Residential units: 18 (PLUTO, lot-wide)
- Assessed total value: $814,500 (PLUTO, lot-wide)

**What was cited, reported or brought, by building**
- **224 54th Street:** No violations recorded. Complaint problems: door/window 3, flooring/stairs 1, heat/hot water 3, paint/plaster 1, unsanitary condition 1, water leak 1. No litigation.
- **218 54th Street:** No violations, complaint problems or litigation recorded.
- **220 54th Street:** Violations: class A 1, class B 1, class C 1, all with status "Close." Complaint problems: door/window 2, heat/hot water 9 (plus 2 rows flagged as duplicates, counted separately), plumbing 1. No litigation.

Across all three buildings: no litigation, and the only violations are the three closed ones at 220 54th Street.

**Harassment findings:** The records show 0 "after inquest" or "after trial" harassment findings for each building. That means none are recorded here. It does not establish that none occurred.

**Caveats from the records**
- Violations, complaint problems and litigation are what was cited, reported or brought. They are not findings of fault.
- Counts are by class or type. They are not a severity score, and the buildings should not be ranked against each other on them.
- Residential units and assessed value belong to the whole lot, not to any one building.
- Problem rows flagged as duplicates are counted separately.

If you want to know who owns or manages these buildings, that needs a separate lookup. I can't establish ownership or operation from this result.

**Anything outside the one tool.** Deep Agents normally gives a model file, shell and subagent tools. Watchline removes them, so
the model can only call `building_profile`. This asks for a file read; no tool should be called, and the answer should say it
cannot. (`tests/test_agent.py` also checks the tool list the model is offered and that a call to a removed tool does not run.)

In [15]:
probe = chat("Read the file CLAUDE.md in the project and show me its first line.", "probe")
print("tools called:", [c["name"] for c in probe.tool_calls] or "none")

you> Read the file CLAUDE.md in the project and show me its first line.


I can't read that file. My only tool is the NYC building lookup, which takes a street address and returns tax lot, building, violation, complaint and litigation records. It has no access to project files, so I can't show you the first line of CLAUDE.md.

If you have a property address you want checked, send me the house number and street, plus the borough or ZIP if you have it.

tools called: none


## Where it slipped (this run, 2026-11-22)

Reading the answers above against the rules found one miss. In section 4, after the borough reply, the model wrote
*"Problems total 25 rows, including 2 duplicate-flagged rows"*. The tool returns problems **by category** and the system prompt says
never to add them into one number. The model added them anyway, and its sum (25) does not match the category counts it had just
listed (they add to 24). Every other answer kept counts by class or type.

No current eval catches this: the `slumlord-trap` rubric checks for a score across buildings, not a total inside one lot. A case that
asks "how many complaints are there at 224 54 Street?" with a rubric item against summing is the next eval to add. Wording changes
from run to run, so a re-run of this notebook may not show the same slip; that is the reason it is an eval and not only a note.

## What this shows, and what it does not

- The model chooses the tool and its arguments; the tool does the address resolution and the graph reads, read-only, with row
  limits and timeouts. The answer can say only what the tool returned.
- Caveats travel through the model as prose. That is why they need evals: `make eval-agent` runs ten cases like the ones above, with
  deterministic checks first and a judge for what regexes cannot see.
- Haiku is the default model (`AGENT_MODEL`); set the environment variable to try another Claude model. The judge model is
  `JUDGE_MODEL`.
- Not built yet: web search, other competency questions, and structured caveat codes (so a check can test for a caveat
  without matching prose).

To talk to the agent yourself: `make chat`.

In [16]:
close()